# Week 5. Modeling IV: logical and discrete decisions, big-M versus convex hull, formulation strength

**Course:** 2105623 Optimization of Chemical Processes
**Institution:** Department of Chemical Engineering, Chulalongkorn University
**Instructor:** Assoc. Prof. Dr. Soorathep Kheawhom

**CLO mapping:** CLO 1 (formulation with discrete decisions), CLO 4 (implementation and solution in Pyomo)

## Learning objectives

By the end of this notebook you should be able to:

- Use binary variables to express yes/no decisions, fixed charges, minimum run rates and semicontinuous variables, and verify that the encoding is exact.
- Translate logical propositions (implication, disjunction, exactly-one, at-most-k, if-then-else) into linear inequalities, and check each translation against its truth table.
- Choose the tightest valid big-M for a constraint, explain what a loose M does to the LP relaxation and to the solver, and demonstrate both effects numerically.
- Build the convex hull (disaggregated) formulation of a disjunction, compare its relaxation with the big-M relaxation in bound, in rows and as a plotted region.
- Report the integrality gap as the measurable definition of formulation strength, and apply the whole toolkit to a process unit selection problem solved end to end.

**Estimated duration:** 110 minutes
**Prerequisites:** Weeks 1 to 4, in particular the blending model of Week 4. Branch and bound is not assumed and is not covered here; it is Week 11.

**Reference:** Williams, *Model Building in Mathematical Programming*, Ch. 9 and 10; Rao, Ch. 10; Balas (1985) on disjunctive programming; Grossmann and Trespalacios (2013) on generalized disjunctive programming.

In [ ]:
# --- Environment check -------------------------------------------------------
import sys, subprocess, importlib, shutil

def ensure(pkg, pip_name=None):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or pkg])

for p, n in [("pyomo", "pyomo"), ("numpy", "numpy"), ("scipy", "scipy"),
             ("matplotlib", "matplotlib"), ("pandas", "pandas")]:
    ensure(p, n)

import numpy as np, pandas as pd, matplotlib.pyplot as plt
import pyomo.environ as pyo

def pick_solver(kind="lp"):
    """Return the first available solver of the requested kind."""
    order = {"lp":   ["appsi_highs", "glpk", "cbc", "gurobi", "cplex"],
             "milp": ["appsi_highs", "cbc", "glpk", "gurobi", "cplex"],
             "nlp":  ["ipopt", "conopt", "knitro"],
             "minlp":["bonmin", "couenne", "mindtpy"]}[kind]
    for name in order:
        try:
            s = pyo.SolverFactory(name)
            if s is not None and s.available(exception_flag=False):
                print(f"Using solver: {name}")
                return s
        except Exception:
            continue
    raise RuntimeError(f"No {kind} solver found. Install one, e.g. 'pip install highspy' "
                       f"or 'conda install -c conda-forge ipopt glpk coincbc'.")

In [ ]:
# --- Make locally installed solvers visible ----------------------------------
# The IDAES extensions install ipopt, bonmin and couenne outside the default PATH.
# This cell is a no-op when the binaries are already on PATH.
import os, time
for extra in [os.path.join(os.path.expanduser("~"), ".idaes", "bin"),
              os.path.join(sys.prefix, "bin")]:
    if os.path.isdir(extra) and extra not in os.environ.get("PATH", "").split(os.pathsep):
        os.environ["PATH"] = os.environ.get("PATH", "") + os.pathsep + extra

available = {name: bool(pyo.SolverFactory(name).available(exception_flag=False))
             for name in ["appsi_highs", "glpk", "cbc", "ipopt"]}
print("solver availability:", available)

In [ ]:
# --- Figure style: Teal-Amber Lab Palette v1.0 -------------------------------
PALETTE = ["#0F6E6B", "#E29A2D", "#BE654C", "#5A91BE", "#83A462", "#995A90", "#333F4A", "#DFC98F"]
INK, GRAPHITE, MIST, PAPER = "#1C242B", "#333F4A", "#B9C1C6", "#F3F0EB"

plt.rcParams.update({
    "figure.dpi": 150, "savefig.dpi": 150,
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.edgecolor": GRAPHITE, "axes.labelcolor": INK, "axes.titlecolor": INK,
    "axes.linewidth": 1.0, "axes.grid": True, "axes.axisbelow": True,
    "grid.color": MIST, "grid.linewidth": 0.7, "grid.alpha": 0.9,
    "xtick.color": GRAPHITE, "ytick.color": GRAPHITE,
    "text.color": INK, "lines.linewidth": 1.8, "lines.markersize": 5,
    "font.size": 9, "legend.frameon": False,
    "axes.prop_cycle": plt.cycler(color=PALETTE),
})

def tidy(ax):
    """Apply the house style to a single Axes object."""
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(GRAPHITE)
    return ax

print("Palette loaded:", ", ".join(PALETTE[:3]), "...")

## 1. Binary variables as modeling devices

A continuous variable answers *how much*. A binary variable answers *whether*. Four patterns cover most of
what a process model needs.

**Yes/no decision.** `z in {0,1}` is 1 if a unit is installed, a stream is lined up, a contract is signed.
Nothing else is required: the variable carries the decision and the objective carries its price.

**Fixed charge.** A cost `F` is incurred if and only if an activity happens at all. Written as `F z` in the
objective, it is linear in `z`, and the link to the continuous activity `u` is

    u <= M z

so that `z = 0` forces `u = 0`. Without that link the solver would set `z = 0` and still use `u > 0`.

**Minimum run rate.** A pump, a compressor or a reactor that runs at all must run above a technical minimum
`L`. Written as

    u >= L z

so that `z = 1` forces `u >= L`, while `z = 0` leaves it vacuous.

**Semicontinuous variable.** The pair of the two constraints above encodes

    u  in  {0}  union  [L, M]

which is a **disjunction**, not an interval. This is the smallest nonconvex set that appears in process
models, and the binary is what makes it expressible in a mixed-integer *linear* program.

### The running instance

The refinery blending problem of Week 4 is used throughout, so that the effect of each discrete restriction
can be read as a change in the blending margin. Three operational realities were ignored in Week 4:

1. Lining up a component stream requires a tank, a pump and a transfer line, at a fixed charge `F_c`
   whether the plant moves 1 tonne or 3,000.
2. A transfer line that is running has a minimum run rate `L_c`. Below it the pump cavitates and the
   metering is unreliable, so operations will not authorize the transfer.
3. The tank farm can line up only a limited number of streams at a time, and some streams depend on others.

Items 1 and 2 are handled in this section, item 3 in section 2.

In [ ]:
# --- Week 4 blending instance, reused verbatim -------------------------------
COMPONENTS = ["Reformate", "FCC_Naphtha", "Alkylate", "Butane", "Straight_Run"]
GRADES = ["Regular", "Premium"]

comp = pd.DataFrame({
    "ron":       [98.0,  92.0,  95.0,  93.0,  68.0],
    "rvp_kpa":   [ 3.0,   6.5,   4.5,  62.0,   9.0],
    "sulfur_ppm":[110.0, 380.0,  15.0,   5.0, 520.0],
    "cost_usd_t":[760.0, 730.0, 805.0, 510.0, 695.0],
    "avail_t":   [3200.0, 4000.0, 2000.0, 900.0, 2600.0],
}, index=COMPONENTS)

grade = pd.DataFrame({
    "ron_min":    [91.0,  95.0],
    "rvp_max":    [ 9.0,   8.5],
    "sulfur_max": [290.0, 190.0],
    "price_usd_t":[742.0, 806.0],
    "vol_min_t":  [4000.0, 2500.0],
    "vol_max_t":  [7000.0, 5000.0],
}, index=GRADES)

# discrete operating data
fixed_charge = pd.Series({"Reformate": 12000.0, "FCC_Naphtha": 9000.0, "Alkylate": 15000.0,
                          "Butane": 4000.0, "Straight_Run": 7000.0}, name="fixed_usd")
min_run = pd.Series({"Reformate": 800.0, "FCC_Naphtha": 1000.0, "Alkylate": 500.0,
                     "Butane": 200.0, "Straight_Run": 600.0}, name="min_run_t")
print(pd.concat([comp.cost_usd_t, comp.avail_t, fixed_charge, min_run], axis=1).to_string())

In [ ]:
# --- Blending model with optional discrete structure -------------------------
def build_blend(discrete=False, logic=False, big_M=None, relax=False,
                max_streams=4, use_cardinality=True, use_implication=True):
    m = pyo.ConcreteModel(name="blending")
    m.C = pyo.Set(initialize=COMPONENTS)
    m.G = pyo.Set(initialize=GRADES)
    m.x = pyo.Var(m.C, m.G, domain=pyo.NonNegativeReals)
    m.y = pyo.Var(m.G, domain=pyo.NonNegativeReals)
    m.u = pyo.Var(m.C, domain=pyo.NonNegativeReals)

    m.grade_balance = pyo.Constraint(m.G, rule=lambda m, g: sum(m.x[c, g] for c in m.C) == m.y[g])
    m.total_use     = pyo.Constraint(m.C, rule=lambda m, c: m.u[c] == sum(m.x[c, g] for g in m.G))
    m.availability  = pyo.Constraint(m.C, rule=lambda m, c: m.u[c] <= comp.avail_t[c])
    m.vol_lo = pyo.Constraint(m.G, rule=lambda m, g: m.y[g] >= grade.vol_min_t[g])
    m.vol_hi = pyo.Constraint(m.G, rule=lambda m, g: m.y[g] <= grade.vol_max_t[g])
    m.octane = pyo.Constraint(m.G, rule=lambda m, g: sum(comp.ron[c]*m.x[c, g] for c in m.C)
                                                     >= grade.ron_min[g]*m.y[g])
    m.vapor  = pyo.Constraint(m.G, rule=lambda m, g: sum(comp.rvp_kpa[c]*m.x[c, g] for c in m.C)
                                                     <= grade.rvp_max[g]*m.y[g])
    m.sulfur = pyo.Constraint(m.G, rule=lambda m, g: sum(comp.sulfur_ppm[c]*m.x[c, g] for c in m.C)
                                                     <= grade.sulfur_max[g]*m.y[g])

    fixed_term = 0
    if discrete:
        m.z = pyo.Var(m.C, domain=pyo.UnitInterval if relax else pyo.Binary)
        M = {c: (comp.avail_t[c] if big_M is None else big_M) for c in COMPONENTS}
        m.on_upper = pyo.Constraint(m.C, rule=lambda m, c: m.u[c] <= M[c]*m.z[c])
        m.on_lower = pyo.Constraint(m.C, rule=lambda m, c: m.u[c] >= min_run[c]*m.z[c])
        fixed_term = sum(fixed_charge[c]*m.z[c] for c in m.C)
        if logic:
            if use_cardinality:
                m.cardinality = pyo.Constraint(expr=sum(m.z[c] for c in m.C) <= max_streams)
            if use_implication:
                m.butane_needs_sr = pyo.Constraint(expr=m.z["Butane"] <= m.z["Straight_Run"])

    m.profit = pyo.Objective(
        expr=sum(grade.price_usd_t[g]*m.y[g] for g in m.G)
             - sum(comp.cost_usd_t[c]*m.u[c] for c in m.C) - fixed_term,
        sense=pyo.maximize)
    return m

lp   = pick_solver("lp")
milp = pick_solver("milp")

m_lp = build_blend(discrete=False)
r_lp = milp.solve(m_lp)
assert r_lp.solver.termination_condition == pyo.TerminationCondition.optimal
PROFIT_LP = pyo.value(m_lp.profit)

m_fc = build_blend(discrete=True)
r_fc = milp.solve(m_fc)
assert r_fc.solver.termination_condition == pyo.TerminationCondition.optimal
PROFIT_FC = pyo.value(m_fc.profit)

print(f"A. LP, no discrete decisions          : {PROFIT_LP:>14,.2f} USD")
print(f"B. + fixed charges and minimum runs   : {PROFIT_FC:>14,.2f} USD")
print(f"   cost of the discrete structure     : {PROFIT_LP - PROFIT_FC:>14,.2f} USD\n")

detail = pd.DataFrame({
    "usage_t":  [round(pyo.value(m_fc.u[c]), 3) for c in COMPONENTS],
    "on":       [round(pyo.value(m_fc.z[c])) for c in COMPONENTS],
    "min_run":  min_run.values,
    "avail":    comp.avail_t.values,
    "fixed":    fixed_charge.values,
    "LP usage": [round(pyo.value(m_lp.u[c]), 3) for c in COMPONENTS],
}, index=COMPONENTS)
print(detail.to_string())

# the semicontinuous encoding must hold exactly
for c in COMPONENTS:
    u_, z_ = pyo.value(m_fc.u[c]), round(pyo.value(m_fc.z[c]))
    assert (z_ == 0 and u_ < 1e-6) or (z_ == 1 and u_ >= min_run[c] - 1e-6), \
        f"semicontinuous logic violated for {c}"
paid = sum(fixed_charge[c]*round(pyo.value(m_fc.z[c])) for c in COMPONENTS)
print(f"\nfixed charges paid                    : {paid:>14,.2f} USD")
print(f"cost attributable to minimum run rates: "
      f"{PROFIT_LP - PROFIT_FC - paid:>14,.2f} USD")
print("semicontinuous encoding verified for every component")

---

### The same model in the spreadsheet

The companion workbook for this week is `w05-fixed-charge.xlsx`. Per the tool allocation table
of the course specification, Week 5 is a **mixed week**: the fixed-charge model below is built in
OpenSolver driving CBC, which is a genuine branch and bound code, and the formulation-strength
comparison of Sections 3 to 5 then moves to Pyomo because it needs dozens of re-solves. The layout
extends the Week 4 blending sheet, so the discrete structure is visible as a set of added columns.

| Algebraic symbol | Spreadsheet range or layout | Pyomo component | Note |
|---|---|---|---|
| `c in C`, components | row labels `A20:A24` | `m.C = pyo.Set(initialize=COMPONENTS)` | Unchanged from Week 4, deliberately: the discrete model is the same sheet with columns added. |
| `g in G`, grades | column headers `C19:D19` | `m.G = pyo.Set(initialize=GRADES)` | |
| `F_c`, fixed charge, USD | data column `H5:H9`, blue text | the `fixed_charge` Series in the `m.profit` expression | |
| `L_c`, minimum run rate, t | data column `I5:I9`, blue text | the `min_run` Series in `m.on_lower` | |
| `M_c`, the big-M | column `I20:I24`, a formula `=F5` copied down so that `M_c = avail_c` | the `big_M` argument of `build_blend`, default `None` meaning `avail_c` | Giving `M` its own visible column is the right habit in either tool. A `1e7` buried inside a formula is invisible in the sheet and invisible in the code. |
| `x_cg >= 0` | changing cells `C20:D24`, green fill | `m.x = pyo.Var(m.C, m.G, domain=pyo.NonNegativeReals)` | |
| `y_g >= 0` | changing cells `C26:D26`, green fill | `m.y = pyo.Var(m.G, domain=pyo.NonNegativeReals)` | |
| `u_c`, total usage of a component | `VALUE` column `E20:E24`, `=SUM(C20:D20)` | `m.u = pyo.Var(m.C, ...)` together with `m.total_use = pyo.Constraint(m.C, ...)` | Pyomo needs a variable plus a defining equality where the sheet needs only a formula cell. The extra row is the price of giving the quantity a name that other constraints can use. |
| `z_c in {0,1}`, stream lined up | changing cells `H20:H24`, green fill, entered in the dialog as `$H$20:$H$24 bin` | `m.z = pyo.Var(m.C, domain=pyo.Binary)` | Integrality is a dialog entry in the sheet and a `domain` in Pyomo. Only the second travels with the model. |
| `u_c <= M_c z_c` | `VALUE` column `J20:J24`, `=E20-I20*H20`; relation column `K20:K24` holding `<=`; `RHS` column `L20:L24` of zeros | `m.on_upper = pyo.Constraint(m.C, rule=...)` | The `VALUE` cell multiplies a data cell by a binary changing cell, so the row is still linear. Keep the engine on CBC; selecting GRG here would be a modeling error, not a preference. |
| `u_c >= L_c z_c` | `VALUE` column `M20:M24`, `=E20-$I5*H20`; relation column `N20:N24` holding `>=`; `RHS` column `O20:O24` of zeros | `m.on_lower = pyo.Constraint(m.C, rule=...)` | The two columns together encode the semicontinuous set `{0} union [L_c, M_c]`. |
| `sum_c z_c <= 4`, cardinality | single cell `H26`, `=SUM(H20:H24)`; relation `I26`; `RHS` `J26` | `m.cardinality = pyo.Constraint(expr=...)` | A scalar constraint is one cell, not a family, in both tools. |
| `z_Butane <= z_Straight_Run`, implication | single cell `H28`, `=H23-H24`; relation `I28` holding `<=`; `RHS` `J28` holding `0` | `m.butane_needs_sr = pyo.Constraint(expr=...)` | The logical proposition of Section 2 is one difference of two changing cells. |
| octane, RVP and sulfur specifications | rows 32 to 42 of the Week 4 sheet, unchanged | `m.octane`, `m.vapor`, `m.sulfur` | The continuous part of the model is untouched by the discrete part, which is exactly why the comparison in Section 1 is meaningful. |
| `max ... - sum_c F_c z_c` | objective cell `C45`, the Week 4 formula with `-SUMPRODUCT($H$5:$H$9,H20:H24)` appended | `m.profit = pyo.Objective(expr=..., sense=pyo.maximize)` | The fixed-charge term is linear in `z`, so the objective cell stays a sum of `SUMPRODUCT`s. |
| the LP relaxation, `z_c in [0,1]` | delete the `bin` entry and add `$H$20:$H$24 <= 1` | `relax=True`, that is `domain=pyo.UnitInterval` | One dialog edit against one keyword. The difference is that the sheet can hold only one of the two results at a time. |
| the disaggregated, convex hull form | a second sheet carrying one copy of the continuous block per disjunct | `build_hull` in Section 4, and `strong=True` in Section 6 | The hull formulation changes the number of variables, so it is a new workbook sheet rather than an edit to this one. |

**Where the spreadsheet stops working.** The fixed-charge blending MILP is a perfectly reasonable
spreadsheet: five binaries added to a model that already fitted, driven by CBC through OpenSolver, and
it returns the margin printed above. What does not fit is the quantity this week is actually about.
Formulation strength is the distance between the integer optimum and the optimum of its relaxation, so
every entry in Sections 3 to 5 is a **pair** of solves rather than one, and the pairs multiply fast:
three values of `M` in Section 3, four formulations of the same disjunction in Section 4, plus the
larger instance, all collected into one comparison in Section 5. In the sheet each of those solves means
retyping the `M` column or deleting the `bin` entry, reopening the dialog, clicking Solve, and copying
the objective cell into a table by hand, with nothing to catch a transcription error and no record of
what was clicked. Two of the compared quantities cannot be produced by the dialog at all: OpenSolver
reports a solution but not a node count, and the relaxation areas drawn in Figure 3 come from 180
support-function LPs per formulation, solved in a loop. Getting to a single figure of this notebook
takes well over a hundred solves, which is why the week starts in the spreadsheet and ends in Python.

---

In [ ]:
# --- Figure 1: the semicontinuous set and what the relaxation sees -----------
fig, axes = plt.subplots(1, 2, figsize=(10.0, 3.7), constrained_layout=True)

cc = "Straight_Run"
L, Mt, Ml = min_run[cc], comp.avail_t[cc], 12000.0
ax = tidy(axes[0])
zz = np.linspace(0, 1, 200)
ax.fill_between(zz, L*zz, Ml*zz, color=PALETTE[7], alpha=0.45,
                label=f"relaxation with a loose M = {Ml:,.0f}")
ax.fill_between(zz, L*zz, Mt*zz, color=PALETTE[0], alpha=0.30,
                label=f"relaxation with the tight M = {Mt:,.0f}")
ax.plot(zz, Mt*zz, color=PALETTE[0], lw=1.6)
ax.plot(zz, Ml*zz, color=PALETTE[1], lw=1.6)
ax.plot(zz, L*zz, color=GRAPHITE, lw=1.4, ls="--", label=f"u = L z, L = {L:,.0f}")
ax.plot([0], [0], marker="o", ms=8, color=PALETTE[2], ls="none",
        label="integer feasible: z = 0, u = 0")
ax.plot([1, 1], [L, Mt], color=PALETTE[2], lw=4.0, solid_capstyle="butt",
        label="integer feasible: z = 1, L <= u <= M")
ax.set_xlabel("binary z (relaxed to [0, 1])"); ax.set_ylabel("stream usage u, tonnes")
ax.set_ylim(0, Ml*1.02); ax.set_xlim(-0.02, 1.05)
ax.set_title(f"Semicontinuous usage of {cc.replace('_', ' ')}", fontsize=10)
ax.legend(fontsize=7.2, loc="upper left")

ax = tidy(axes[1])
labels = ["LP\noptimum", "fixed charge\n+ minimum run"]
vals = [PROFIT_LP, PROFIT_FC]
bars = ax.bar(labels, vals, color=[PALETTE[0], PALETTE[1]], width=0.55)
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width()/2, v + 8000, f"{v:,.0f}", ha="center", fontsize=8.5, color=INK)
ax.annotate("", xy=(0.5, vals[1]), xytext=(0.5, vals[0]),
            arrowprops=dict(arrowstyle="<->", color=GRAPHITE, lw=1.2))
ax.text(0.55, 0.5*(vals[0] + vals[1]), f"-{vals[0]-vals[1]:,.0f} USD",
        fontsize=8, color=PALETTE[2], va="center", ha="left")
ax.set_ylabel("blending margin, USD"); ax.set_ylim(0, max(vals)*1.2)
ax.set_title("What the discrete structure costs", fontsize=10)

fig.suptitle("Week 5, Figure 1: a binary turns a disjunction into a linear model",
             color=INK, fontsize=10)
plt.show()

## 2. Logical propositions as linear constraints

Let `z_a, z_b, z_j` be binary variables that are 1 when the corresponding proposition is true. Every
propositional statement over a finite set of such variables can be written as linear inequalities. The
translations below are the ones worth memorizing.

### Translation table

| Statement in words | Logic | Linear constraint |
|---|---|---|
| a implies b (if a then b) | `a => b` | `z_a <= z_b` |
| a and b are equivalent | `a <=> b` | `z_a = z_b` |
| at least one of a, b | `a or b` | `z_a + z_b >= 1` |
| not both | `not (a and b)` | `z_a + z_b <= 1` |
| exactly one of a set S | | `sum_{j in S} z_j = 1` |
| at least k of S | | `sum_{j in S} z_j >= k` |
| at most k of S | | `sum_{j in S} z_j <= k` |
| a and b together imply c | `a and b => c` | `z_a + z_b - 1 <= z_c` |
| a implies b or c | `a => b or c` | `z_a <= z_b + z_c` |
| c is true exactly when a and b are | `c <=> a and b` | `z_c <= z_a`, `z_c <= z_b`, `z_c >= z_a + z_b - 1` |
| c is true exactly when a or b is | `c <=> a or b` | `z_c >= z_a`, `z_c >= z_b`, `z_c <= z_a + z_b` |
| if a then `u <= U1`, else `u <= U2` | if-then-else | `u <= U1 z_a + U2 (1 - z_a)` |
| if a then `g(x) <= 0` | conditional constraint | `g(x) <= M (1 - z_a)` |

The last two rows are where continuous variables enter, and where the constant `M` appears. Everything above
them is exact and dimensionless; everything below them depends on a numerical constant whose choice is the
subject of section 3.

A translation is either right or wrong, and the check is mechanical: enumerate every assignment of the
binaries, evaluate the proposition and evaluate the inequality, and confirm that the two agree on every row.
The next cell does exactly that for each line of the table.

In [ ]:
# --- Verify every translation against its truth table ------------------------
import itertools

def check(name, n_vars, proposition, constraints):
    """Enumerate all 2**n_vars assignments and compare the proposition with the constraints."""
    rows, ok = [], True
    for assign in itertools.product([0, 1], repeat=n_vars):
        want = bool(proposition(*assign))
        got = all(cons(*assign) for cons in constraints)
        ok &= (want == got)
        rows.append(list(assign) + [int(want), int(got)])
    return {"statement": name, "assignments": 2**n_vars, "agrees": ok}, rows

tests = [
    check("a => b", 2, lambda a, b: (not a) or b, [lambda a, b: a <= b]),
    check("a <=> b", 2, lambda a, b: a == b, [lambda a, b: a == b]),
    check("a or b", 2, lambda a, b: a or b, [lambda a, b: a + b >= 1]),
    check("not both", 2, lambda a, b: not (a and b), [lambda a, b: a + b <= 1]),
    check("exactly one of a, b, c", 3, lambda a, b, c: a + b + c == 1,
          [lambda a, b, c: a + b + c == 1]),
    check("at most 2 of a, b, c", 3, lambda a, b, c: a + b + c <= 2,
          [lambda a, b, c: a + b + c <= 2]),
    check("a and b => c", 3, lambda a, b, c: (not (a and b)) or c,
          [lambda a, b, c: a + b - 1 <= c]),
    check("a => b or c", 3, lambda a, b, c: (not a) or b or c,
          [lambda a, b, c: a <= b + c]),
    check("c <=> a and b", 3, lambda a, b, c: bool(c) == bool(a and b),
          [lambda a, b, c: c <= a, lambda a, b, c: c <= b, lambda a, b, c: c >= a + b - 1]),
    check("c <=> a or b", 3, lambda a, b, c: bool(c) == bool(a or b),
          [lambda a, b, c: c >= a, lambda a, b, c: c >= b, lambda a, b, c: c <= a + b]),
]
table = pd.DataFrame([t[0] for t in tests])
print(table.to_string(index=False))
assert table["agrees"].all(), "at least one translation does not match its truth table"

print("\ntruth table for  c <=> a and b  (the one students most often get wrong)")
tt = pd.DataFrame([r for r in tests[8][1]], columns=["z_a", "z_b", "z_c", "proposition", "constraints"])
print(tt.to_string(index=False))

# a deliberately wrong translation must fail the same test
bad, _ = check("a => b written as z_a + z_b <= 1 (WRONG)", 2,
               lambda a, b: (not a) or b, [lambda a, b: a + b <= 1])
print(f"\ncontrol: {bad['statement']} agrees with the truth table: {bad['agrees']}")
assert not bad["agrees"], "the control case should fail"

In [ ]:
# --- Two logical conditions on the blending problem --------------------------
# Cardinality: the tank farm can line up at most four streams at once.
#     sum_c z_c <= 4
# Implication: butane may only move while the straight-run splitter runs, because
# they share the vapor recovery header:   z_Butane => z_Straight_Run,  i.e.  z_B <= z_SR
cases = {}
for label, kw in [("B. fixed charge and min run", dict(discrete=True, logic=False)),
                  ("C1. + cardinality only", dict(discrete=True, logic=True,
                                                  use_implication=False)),
                  ("C2. + implication only", dict(discrete=True, logic=True,
                                                  use_cardinality=False)),
                  ("C3. + both", dict(discrete=True, logic=True))]:
    mm = build_blend(**kw)
    rr = milp.solve(mm)
    assert rr.solver.termination_condition == pyo.TerminationCondition.optimal, label
    cases[label] = (pyo.value(mm.profit), {c: round(pyo.value(mm.z[c])) for c in COMPONENTS}, mm)

for label, (v, z, _) in cases.items():
    on = ", ".join(c for c in COMPONENTS if z[c] == 1)
    print(f"{label:<30s} {v:>13,.2f} USD   streams on: {on}")

m_full = cases["C3. + both"][2]
PROFIT_LOGIC = cases["C3. + both"][0]
def cost_of(label):
    d = cases["B. fixed charge and min run"][0] - cases[label][0]
    return 0.0 if abs(d) < 5e-3 else d

print(f"\ncost of the cardinality limit alone      : {cost_of('C1. + cardinality only'):>12,.2f} USD")
print(f"cost of the implication alone            : {cost_of('C2. + implication only'):>12,.2f} USD")
print(f"cost of both together                    : {cost_of('C3. + both'):>12,.2f} USD")
print(f"cost of adding the implication on top of "
      f"cardinality: {cases['C1. + cardinality only'][0] - PROFIT_LOGIC:>12,.2f} USD")
print(f"total cost of discrete reality      : {PROFIT_LP - PROFIT_LOGIC:>12,.2f} USD "
      f"({100*(PROFIT_LP - PROFIT_LOGIC)/PROFIT_LP:.1f} percent of the LP margin)")

assert sum(round(pyo.value(m_full.z[c])) for c in COMPONENTS) <= 4
assert round(pyo.value(m_full.z["Butane"])) <= round(pyo.value(m_full.z["Straight_Run"]))
print("\ncardinality and implication both verified at the optimum")
print("grade volumes:", {g: round(pyo.value(m_full.y[g]), 3) for g in GRADES})

## 3. Big-M formulations

A constraint that applies only when a binary is 1 is written

    g(x)  <=  M ( 1 - z )

so that `z = 1` enforces `g(x) <= 0` and `z = 0` leaves the row slack. Any `M` large enough to make the row
non-binding when `z = 0` is **valid**. Only the smallest such value is **tight**.

### How to choose M

The tightest valid value is the largest amount by which the constraint can be violated over the rest of the
feasible set:

    M*  =  max { g(x)  :  x feasible for the other alternatives }

which is itself an optimization problem, usually an LP. Three practical recipes, in decreasing order of
quality:

1. Solve the little LP above. This is what a good modeling layer does automatically.
2. Use variable bounds: if `g(x) = a'x - b` and `xL <= x <= xU`, then
   `M = sum_i max(a_i xU_i, a_i xL_i) - b` is valid.
3. Use a physical cap that is already in the model. In the blending problem the availability limit
   `avail_c` caps `u_c`, so `M_c = avail_c` is both valid and tight for `u_c <= M_c z_c`, and `L_c` is
   already tight for `u_c >= L_c z_c`.

What must never be done is to pick a round number such as `10^6` because it "looks big enough". Two things
go wrong, and they are different failures.

### Failure 1: the relaxation collapses

Relaxing `z` to `[0, 1]` in `u <= M z` gives `z >= u / M`. With a tight `M` a stream that is used at all
forces `z` close to 1 and the relaxation pays most of the fixed charge. With `M` a hundred times larger, the
same usage forces `z` to only a hundredth of that, so the relaxation buys the stream for one percent of its
fixed charge. The bound becomes useless, and the bound is what a branch-and-bound search spends its effort
closing.

### Failure 2: the integrality tolerance becomes a hole in the model

Solvers accept `z` as integral when it is within an absolute tolerance (typically `10^-6`) of 0 or 1. A
value `z = 10^-6` is therefore reported as `z = 0`, while `u <= M z` still allows `u` up to `M * 10^-6`.
With `M = 10^10` that is 10,000 tonnes of flow through a stream the solution claims is switched off, and the
fixed charge paid is `F * 10^-6`, a fraction of a cent. The model is not wrong in exact arithmetic. It is
wrong in floating-point arithmetic, which is the only kind a solver has.

In [ ]:
# --- Failure 1 on the blending model: same optimum, much weaker bound --------
rows = []
for label, M in [("tight  M_c = avail_c", None), ("loose  M = 50,000", 50000.0),
                 ("very loose  M = 10^7", 1e7)]:
    m_int = build_blend(discrete=True, logic=True, big_M=M)
    r_int = milp.solve(m_int)
    assert r_int.solver.termination_condition == pyo.TerminationCondition.optimal
    m_rel = build_blend(discrete=True, logic=True, big_M=M, relax=True)
    r_rel = lp.solve(m_rel)
    assert r_rel.solver.termination_condition == pyo.TerminationCondition.optimal
    rows.append({"formulation": label,
                 "rows": m_int.nconstraints(),
                 "MILP optimum": round(pyo.value(m_int.profit), 2),
                 "LP relaxation bound": round(pyo.value(m_rel.profit), 2),
                 "gap USD": round(pyo.value(m_rel.profit) - pyo.value(m_int.profit), 2),
                 "gap percent": round(100*(pyo.value(m_rel.profit) - pyo.value(m_int.profit))
                                      / pyo.value(m_int.profit), 2),
                 "sum of relaxed z": round(sum(pyo.value(m_rel.z[c]) for c in COMPONENTS), 4)})
bigM = pd.DataFrame(rows).set_index("formulation")
print(bigM.to_string())
assert abs(bigM["MILP optimum"].max() - bigM["MILP optimum"].min()) < 1e-4, \
    "every valid big-M must give the same integer optimum"
assert bigM["LP relaxation bound"].iloc[1] > bigM["LP relaxation bound"].iloc[0] + 1.0, \
    "the loose formulation must give a weaker bound"
print("\nSame integer optimum, weaker bound. A loose M does not change the answer, "
      "it changes how long the search takes to prove it.")

In [ ]:
# --- Failure 2: what the integrality tolerance buys at each value of M -------
INT_TOL = 1e-6                      # the default absolute integrality tolerance of most solvers
hole = pd.DataFrame({
    "M": [3200.0, 5e4, 1e7, 1e10],
    "flow allowed while z is reported as 0 (t)": [M*INT_TOL for M in [3200.0, 5e4, 1e7, 1e10]],
    "fixed charge actually paid (USD)": [12000.0*INT_TOL]*4,
})
print(hole.to_string(index=False, float_format=lambda v: f"{v:,.6g}"))

# make it concrete: fix every binary at the integrality tolerance and solve the LP that remains
m_hole = build_blend(discrete=True, big_M=1e10, relax=True)
for c in COMPONENTS:
    m_hole.z[c].fix(INT_TOL)
r_hole = lp.solve(m_hole)
assert r_hole.solver.termination_condition == pyo.TerminationCondition.optimal
paid_hole = sum(fixed_charge[c]*INT_TOL for c in COMPONENTS)
print(f"\nWith M = 10^10 and every z fixed at {INT_TOL:g} (which the solver reports as z = 0):")
print(f"  margin obtained            : {pyo.value(m_hole.profit):>14,.4f} USD")
print(f"  fixed charges actually paid: {paid_hole:>14,.4f} USD "
      f"(against {fixed_charge.sum():,.0f} USD if the streams are honestly switched on)")
print(f"  true MILP optimum          : {PROFIT_FC:>14,.4f} USD")
assert pyo.value(m_hole.profit) > PROFIT_FC + 1000.0, \
    "the pathological point should look better than the true optimum"
print("\nThe reported plan is infeasible in reality and the solver is not at fault: "
      "M * tolerance is a modeling decision.")

In [ ]:
# --- A larger instance where the bound also costs solve time -----------------
# 20 candidate process units, 40 production orders. An order may be split across units.
# Each unit that is used at all incurs an annual fixed charge.
rng = np.random.default_rng(3)
NU, NO = 20, 40
units_s  = [f"U{j+1:02d}" for j in range(NU)]
orders_s = [f"O{i+1:02d}" for i in range(NO)]
size = {i: float(rng.integers(8, 25)) for i in orders_s}
TOTAL = sum(size.values())
cap  = {j: float(np.round(2.0*TOTAL/NU, 1)) for j in units_s}       # t/yr on each unit
fixc = {j: float(np.round(rng.uniform(200, 400), 1)) for j in units_s}   # kUSD/yr
vc   = {(i, j): float(np.round(rng.uniform(1, 9), 2)) for i in orders_s for j in units_s}
print(f"total order volume {TOTAL:,.0f} t/yr, capacity of each unit {cap[units_s[0]]:,.1f} t/yr, "
      f"so at least {int(np.ceil(TOTAL/cap[units_s[0]]))} units are needed")

def build_alloc(kind, relax=False):
    """kind in {'loose', 'tight', 'hull'} selects the formulation of the on/off link."""
    m = pyo.ConcreteModel(name=f"allocation_{kind}")
    m.I = pyo.Set(initialize=orders_s)
    m.J = pyo.Set(initialize=units_s)
    m.x = pyo.Var(m.I, m.J, domain=pyo.NonNegativeReals)
    m.z = pyo.Var(m.J, domain=pyo.UnitInterval if relax else pyo.Binary)
    m.assign = pyo.Constraint(m.I, rule=lambda m, i: sum(m.x[i, j] for j in m.J) == size[i])
    Mj = {j: (1e4 if kind == "loose" else cap[j]) for j in units_s}
    m.switch = pyo.Constraint(m.J, rule=lambda m, j: sum(m.x[i, j] for i in m.I) <= Mj[j]*m.z[j])
    if kind == "loose":            # the physical capacity must still be imposed separately
        m.capacity = pyo.Constraint(m.J, rule=lambda m, j: sum(m.x[i, j] for i in m.I) <= cap[j])
    if kind == "hull":             # disaggregated (hull-derived) variable upper bounds
        m.link = pyo.Constraint(m.I, m.J, rule=lambda m, i, j: m.x[i, j] <= size[i]*m.z[j])
    m.cost = pyo.Objective(expr=sum(fixc[j]*m.z[j] for j in m.J)
                           + sum(vc[i, j]*m.x[i, j] for i in m.I for j in m.J),
                           sense=pyo.minimize)
    return m

strength = []
for kind, label in [("loose", "big-M, M = 10,000"), ("tight", "big-M, M = capacity"),
                    ("hull", "disaggregated (hull-derived)")]:
    mi = build_alloc(kind)
    t0 = time.perf_counter(); ri = milp.solve(mi); t_milp = time.perf_counter() - t0
    assert ri.solver.termination_condition == pyo.TerminationCondition.optimal, label
    ml = build_alloc(kind, relax=True)
    t1 = time.perf_counter(); rl = lp.solve(ml); t_lp = time.perf_counter() - t1
    assert rl.solver.termination_condition == pyo.TerminationCondition.optimal
    z_milp, z_lp = pyo.value(mi.cost), pyo.value(ml.cost)
    strength.append({"formulation": label, "rows": mi.nconstraints(), "cols": mi.nvariables(),
                     "MILP optimum": round(z_milp, 3), "LP bound": round(z_lp, 3),
                     "integrality gap percent": round(100*(z_milp - z_lp)/z_milp, 4),
                     "MILP time s": round(t_milp, 3), "LP time s": round(t_lp, 4),
                     "units opened": sum(1 for j in units_s if pyo.value(mi.z[j]) > 0.5)})
strength = pd.DataFrame(strength)
print()
print(strength.to_string(index=False))
assert abs(strength["MILP optimum"].max() - strength["MILP optimum"].min()) < 1e-3, \
    "all three formulations describe the same integer problem"
assert strength["LP bound"].iloc[2] > strength["LP bound"].iloc[1] > strength["LP bound"].iloc[0], \
    "bounds should improve from loose to tight to disaggregated"

In [ ]:
# --- Figure 2: bound quality and its price in solve time ---------------------
fig, axes = plt.subplots(1, 2, figsize=(10.0, 3.7), constrained_layout=True)
labels2 = ["big-M\nM = 10,000", "big-M\nM = capacity", "disaggregated\n(hull-derived)"]
xpos = np.arange(3)

ax = tidy(axes[0])
ax.bar(xpos - 0.19, strength["LP bound"], width=0.36, color=PALETTE[1], label="LP relaxation bound")
ax.bar(xpos + 0.19, strength["MILP optimum"], width=0.36, color=PALETTE[0], label="MILP optimum")
for k in range(3):
    ax.text(xpos[k], strength["MILP optimum"][k]*1.06,
            f"gap {strength['integrality gap percent'][k]:.3g} %", ha="center",
            fontsize=8, color=PALETTE[2])
ax.set_xticks(xpos); ax.set_xticklabels(labels2, fontsize=8)
ax.set_ylabel("annual cost, kUSD"); ax.set_ylim(0, strength["MILP optimum"].max()*1.38)
ax.set_title("A weak bound is the whole cost of a lazy M", fontsize=10)
ax.legend(fontsize=8, loc="upper left")

ax = tidy(axes[1])
bars = ax.bar(xpos, strength["MILP time s"], color=[PALETTE[2], PALETTE[0], PALETTE[4]], width=0.55)
for b, v in zip(bars, strength["MILP time s"]):
    ax.text(b.get_x() + b.get_width()/2, v*1.03 + 0.02, f"{v:.2f} s", ha="center",
            fontsize=8.5, color=INK)
ax.set_xticks(xpos); ax.set_xticklabels(labels2, fontsize=8)
ax.set_ylabel("time to prove optimality, s")
ax.set_ylim(0, max(strength["MILP time s"])*1.25)
ax.set_title(f"Same answer ({strength['MILP optimum'][0]:,.0f} kUSD), different effort", fontsize=10)

fig.suptitle("Week 5, Figure 2: 20 candidate units, 40 orders, three formulations of the same model",
             color=INK, fontsize=10)
plt.show()

## 4. Convex hull versus big-M

A **disjunction** states that `x` must satisfy one of several alternative sets of constraints:

    OR_k  [ Y_k ;  A_k x <= b_k ]        exactly one Y_k is true

Two standard ways exist to write this as a mixed-integer linear program.

**Big-M.** Keep one copy of `x` and relax the inactive alternatives:

    A_k x <= b_k + M_k ( 1 - z_k ),      sum_k z_k = 1,     z_k in {0,1}

Compact: no new continuous variables, one row per original row. Weak: when `z` is relaxed to `[0,1]`, a
fractional `z` relaxes *every* alternative at once by a large amount.

**Convex hull, or disaggregated form (Balas).** Give each alternative its own copy of the variables and let
the binary scale the copy:

    x = sum_k x_k,     A_k x_k <= b_k z_k,     0 <= x_k <= x^U z_k,     sum_k z_k = 1

When `z_k` is relaxed to `[0,1]`, the projection of this system onto `x` is exactly the convex hull of the
union of the alternatives, which is the tightest linear relaxation that any formulation of the disjunction
can have. The price is `K` copies of the continuous variables and more rows.

### The instance used below

A reactor section can be built in one of two configurations, with `x1` the feed rate (t/h) and `x2` the
product rate (t/h):

- **Configuration A**, a small continuous unit: `10 <= x1 <= 40` with a yield band `0.3 x1 <= x2 <= 0.5 x1`,
  annualized fixed cost 120 kUSD.
- **Configuration B**, a large unit: `60 <= x1 <= 90` with a better yield band
  `0.5 x1 <= x2 <= 0.7 x1`, annualized fixed cost 260 kUSD.

Operating cost is 1.8 kUSD per unit of feed and 3.2 kUSD per unit of product in either configuration, and the
section must deliver at least 25 units of product.

The relaxation regions are drawn by computing the **support function** of each relaxed feasible set: for 180
directions `d` the LP `max d'x` is solved and the optimal `x` recorded. The convex hull of the recorded points
is the projection of the relaxation onto the `(x1, x2)` plane, obtained without any grid sampling.

In [ ]:
# --- The two-configuration disjunction, both formulations --------------------
MODES = ["A", "B"]
# each alternative as rows a1*x1 + a2*x2 <= b
DISJ = {"A": [(-1.0, 0.0, -10.0), (1.0, 0.0, 40.0), (-0.5, 1.0, 0.0), (0.3, -1.0, 0.0)],
        "B": [(-1.0, 0.0, -60.0), (1.0, 0.0, 90.0), (-0.7, 1.0, 0.0), (0.5, -1.0, 0.0)]}
FIXK = {"A": 120.0, "B": 260.0}
VARC = (1.8, 3.2)
XUB  = (120.0, 80.0)
PRODUCT_MIN = 25.0

def tightest_big_M():
    """M for a row of alternative k is the largest violation of that row over the other alternatives."""
    M = {}
    for k in MODES:
        for r, (a1, a2, b) in enumerate(DISJ[k]):
            worst = 0.0
            for j in MODES:
                if j == k:
                    continue
                sub = pyo.ConcreteModel()
                sub.x = pyo.Var([1, 2], bounds=lambda m, d: (0.0, XUB[d-1]))
                sub.c = pyo.ConstraintList()
                for (c1, c2, d0) in DISJ[j]:
                    sub.c.add(c1*sub.x[1] + c2*sub.x[2] <= d0)
                sub.o = pyo.Objective(expr=a1*sub.x[1] + a2*sub.x[2] - b, sense=pyo.maximize)
                rs = lp.solve(sub)
                assert rs.solver.termination_condition == pyo.TerminationCondition.optimal
                worst = max(worst, pyo.value(sub.o))
            M[k, r] = max(0.0, worst)
    return M

M_TIGHT = tightest_big_M()
print("row-wise tightest M (0 means the row is already implied by the other alternative):")
print({f"{k}, row {r}": round(v, 3) for (k, r), v in M_TIGHT.items()})

def build_bigm(M=None, relax=False, with_objective=True):
    m = pyo.ConcreteModel(name="disjunction_bigM")
    m.x = pyo.Var([1, 2], bounds=lambda m, d: (0.0, XUB[d-1]))
    m.z = pyo.Var(MODES, domain=pyo.UnitInterval if relax else pyo.Binary)
    m.pick = pyo.Constraint(expr=sum(m.z[k] for k in MODES) == 1)
    m.c = pyo.ConstraintList()
    for k in MODES:
        for r, (a1, a2, b) in enumerate(DISJ[k]):
            Mv = M if M is not None else M_TIGHT[k, r]
            m.c.add(a1*m.x[1] + a2*m.x[2] <= b + Mv*(1 - m.z[k]))
    if with_objective:
        m.demand = pyo.Constraint(expr=m.x[2] >= PRODUCT_MIN)
        m.cost = pyo.Objective(expr=sum(FIXK[k]*m.z[k] for k in MODES)
                               + VARC[0]*m.x[1] + VARC[1]*m.x[2], sense=pyo.minimize)
    return m

def build_hull(relax=False, with_objective=True):
    m = pyo.ConcreteModel(name="disjunction_hull")
    m.x = pyo.Var([1, 2], bounds=lambda m, d: (0.0, XUB[d-1]))
    m.z = pyo.Var(MODES, domain=pyo.UnitInterval if relax else pyo.Binary)
    m.xd = pyo.Var(MODES, [1, 2], domain=pyo.NonNegativeReals)      # one copy per alternative
    m.pick = pyo.Constraint(expr=sum(m.z[k] for k in MODES) == 1)
    m.split = pyo.Constraint([1, 2], rule=lambda m, d: m.x[d] == sum(m.xd[k, d] for k in MODES))
    m.c = pyo.ConstraintList()
    for k in MODES:
        for (a1, a2, b) in DISJ[k]:
            m.c.add(a1*m.xd[k, 1] + a2*m.xd[k, 2] <= b*m.z[k])
        for d in (1, 2):
            m.c.add(m.xd[k, d] <= XUB[d-1]*m.z[k])
    if with_objective:
        m.demand = pyo.Constraint(expr=m.x[2] >= PRODUCT_MIN)
        m.cost = pyo.Objective(expr=sum(FIXK[k]*m.z[k] for k in MODES)
                               + VARC[0]*m.x[1] + VARC[1]*m.x[2], sense=pyo.minimize)
    return m

def relaxation_region(builder, n_dir=180, **kw):
    """Trace the projection of a relaxed feasible set by solving max d'x in n_dir directions."""
    m = builder(relax=True, with_objective=False, **kw)
    m.support = pyo.Objective(expr=m.x[1], sense=pyo.maximize)
    pts = []
    for theta in np.linspace(0, 2*np.pi, n_dir, endpoint=False):
        m.support.set_value(np.cos(theta)*m.x[1] + np.sin(theta)*m.x[2])
        rs = lp.solve(m)
        assert rs.solver.termination_condition == pyo.TerminationCondition.optimal
        pts.append((pyo.value(m.x[1]), pyo.value(m.x[2])))
    return np.array(pts)

def polygon_area(P):
    x, y = P[:, 0], P[:, 1]
    return 0.5*abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))

REGIONS = {"big-M, M = 500": relaxation_region(build_bigm, M=500.0),
           "big-M, M = 120 (variable bounds)": relaxation_region(build_bigm, M=120.0),
           "big-M, row-wise tightest M": relaxation_region(build_bigm, M=None),
           "convex hull (disaggregated)": relaxation_region(build_hull)}
for name, P in REGIONS.items():
    print(f"{name:36s} relaxation area {polygon_area(P):10.2f}")

In [ ]:
# --- Figure 3: the relaxation regions of the same disjunction ----------------
def alternative_polygon(k, n_dir=240):
    m = pyo.ConcreteModel()
    m.x = pyo.Var([1, 2], bounds=lambda m, d: (0.0, XUB[d-1]))
    m.c = pyo.ConstraintList()
    for (a1, a2, b) in DISJ[k]:
        m.c.add(a1*m.x[1] + a2*m.x[2] <= b)
    m.o = pyo.Objective(expr=m.x[1], sense=pyo.maximize)
    pts = []
    for theta in np.linspace(0, 2*np.pi, n_dir, endpoint=False):
        m.o.set_value(np.cos(theta)*m.x[1] + np.sin(theta)*m.x[2])
        rs = lp.solve(m)
        assert rs.solver.termination_condition == pyo.TerminationCondition.optimal
        pts.append((pyo.value(m.x[1]), pyo.value(m.x[2])))
    return np.array(pts)

POLY = {k: alternative_polygon(k) for k in MODES}
ORDER = ["big-M, M = 500", "big-M, M = 120 (variable bounds)",
         "big-M, row-wise tightest M", "convex hull (disaggregated)"]
SHADE = [PALETTE[7], PALETTE[3], PALETTE[1], PALETTE[0]]

def draw(ax, names):
    for name in names:
        col = SHADE[ORDER.index(name)]
        P = REGIONS[name]
        ax.fill(P[:, 0], P[:, 1], color=col, alpha=0.28, zorder=1)
        ax.plot(np.append(P[:, 0], P[0, 0]), np.append(P[:, 1], P[0, 1]), color=col, lw=2.0,
                zorder=2, label=f"{name}, area {polygon_area(P):,.0f}")
    for k in MODES:
        P = POLY[k]
        ax.fill(P[:, 0], P[:, 1], color=PALETTE[2], alpha=0.9, zorder=3)
    ax.plot([], [], color=PALETTE[2], lw=8, label="the two configurations (true feasible set)")

fig, axes = plt.subplots(1, 2, figsize=(10.6, 4.4), constrained_layout=True)

ax = tidy(axes[0])
draw(ax, ORDER)
ax.set_xlim(0, 122); ax.set_ylim(0, 82)
ax.set_xlabel("x1, feed rate"); ax.set_ylabel("x2, product rate")
ax.set_title("Four relaxations of one disjunction", fontsize=10)
ax.legend(fontsize=7.0, loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=1)
ax.add_patch(plt.Rectangle((5, 0), 40, 25, fill=False, ec=GRAPHITE, lw=1.1, ls=":", zorder=6))
ax.text(46, 2, "detail", fontsize=7.5, color=GRAPHITE)

ax = tidy(axes[1])
draw(ax, ["big-M, row-wise tightest M", "convex hull (disaggregated)"])
ax.set_xlim(5, 45); ax.set_ylim(0, 25)
ax.set_xlabel("x1, feed rate")
ax.set_title("Detail: the tightest big-M still exceeds the hull", fontsize=10)
ax.annotate("vertex (22.5, 15.75) is in the big-M\nrelaxation but not in the hull",
            xy=(22.5, 15.75), xytext=(24.0, 21.0), fontsize=7.5, color=INK,
            arrowprops=dict(arrowstyle="->", color=GRAPHITE, lw=1.1))
ax.legend(fontsize=7.0, loc="lower right")

fig.suptitle("Week 5, Figure 3: relaxation regions of a two-configuration reactor",
             color=INK, fontsize=10)
plt.show()

In [ ]:
# --- Bound, rows and area for each formulation of the disjunction ------------
disj_rows = []
for label, builder, kw, key in [
        ("big-M, M = 500", build_bigm, dict(M=500.0), "big-M, M = 500"),
        ("big-M, M = 120", build_bigm, dict(M=120.0), "big-M, M = 120 (variable bounds)"),
        ("big-M, row-wise tightest", build_bigm, dict(M=None), "big-M, row-wise tightest M"),
        ("convex hull (disaggregated)", build_hull, {}, "convex hull (disaggregated)")]:
    mi = builder(relax=False, **kw)
    ri = milp.solve(mi)
    assert ri.solver.termination_condition == pyo.TerminationCondition.optimal, label
    ml = builder(relax=True, **kw)
    rl = lp.solve(ml)
    assert rl.solver.termination_condition == pyo.TerminationCondition.optimal
    zi, zl = pyo.value(mi.cost), pyo.value(ml.cost)
    disj_rows.append({"formulation": label, "rows": mi.nconstraints(), "cols": mi.nvariables(),
                      "MILP optimum": round(zi, 4), "LP bound": round(zl, 4),
                      "integrality gap percent": round(100*(zi - zl)/zi, 3),
                      "relaxation area": round(polygon_area(REGIONS[key]), 2)})
disj = pd.DataFrame(disj_rows)
print(disj.to_string(index=False))
assert abs(disj["MILP optimum"].max() - disj["MILP optimum"].min()) < 1e-6, \
    "all four formulations describe the same integer problem"
assert disj["relaxation area"].iloc[3] <= disj["relaxation area"].iloc[2] + 1e-6, \
    "the hull relaxation cannot be larger than any big-M relaxation"
print("\nThe hull region is contained in every big-M region, and the ordering of the areas "
      "matches the ordering of the bounds.")

## 5. Formulation strength as a measurable quantity

Two formulations of the same discrete problem have the same integer optimum by definition. They differ in
their **LP relaxation**, and the difference is measurable:

    integrality gap  =  ( z_MILP - z_LP ) / z_MILP        for a minimization problem
    integrality gap  =  ( z_LP - z_MILP ) / z_MILP        for a maximization problem

A formulation `F1` **dominates** `F2` if the feasible set of its relaxation is contained in that of `F2` for
every instance. Domination implies a gap that is never worse, and usually strictly better. The convex hull
formulation dominates every big-M formulation of the same disjunction, because its projection is the smallest
convex set containing the alternatives.

The gap is not a curiosity. Branch and bound terminates when the incumbent and the best remaining bound meet,
so the work it must do is roughly a function of how far apart they start. This is why an extra thousand rows
that halve the gap are usually a bargain, and why the number of rows is a poor proxy for difficulty.

The table below collects the gap for every formulation built in this notebook.

In [ ]:
# --- All formulations, one measure -------------------------------------------
summary = []
for _, r_ in strength.iterrows():
    summary.append({"instance": "20 units, 40 orders", "formulation": r_["formulation"],
                    "rows": r_["rows"], "LP bound": r_["LP bound"],
                    "MILP optimum": r_["MILP optimum"],
                    "integrality gap percent": r_["integrality gap percent"]})
for _, r_ in disj.iterrows():
    summary.append({"instance": "two-configuration reactor", "formulation": r_["formulation"],
                    "rows": r_["rows"], "LP bound": r_["LP bound"],
                    "MILP optimum": r_["MILP optimum"],
                    "integrality gap percent": r_["integrality gap percent"]})
for label in bigM.index:
    z_i, z_l = bigM.loc[label, "MILP optimum"], bigM.loc[label, "LP relaxation bound"]
    summary.append({"instance": "blending with fixed charges", "formulation": label,
                    "rows": bigM.loc[label, "rows"], "LP bound": z_l, "MILP optimum": z_i,
                    "integrality gap percent": round(100*abs(z_l - z_i)/abs(z_i), 4)})
summary = pd.DataFrame(summary)
print(summary.to_string(index=False))
print("\nSmaller is better in the last column, and it is the only column that predicts solve time.")

## 6. Application: process unit selection with fixed charges and minimum throughputs

A zinc-air battery materials plant must supply three intermediate products. Six candidate process units are
offered, each with an annualized fixed cost, a limited number of operating hours per year, a production rate
that depends on the product, a variable processing cost, and a minimum annual throughput below which the
vendor will not warrant the equipment.

### Model

Sets: units `u`, products `p`, and the pairs `(u, p)` that are technically possible.

Variables: `x_up >= 0` tonnes per year of product `p` made on unit `u`, and `z_u in {0,1}` equal to 1 if unit
`u` is installed.

    min   sum_u F_u z_u  +  sum_(u,p) c_up x_up

    s.t.  sum_u x_up = d_p                                  for all p   (demand)
          sum_p x_up / r_up  <=  H_u z_u                    for all u   (operating hours, big-M with M = H_u)
          sum_p x_up  >=  L_u z_u                           for all u   (minimum throughput)
          x_up  <=  min(d_p, H_u r_up) z_u                  for all u,p (disaggregated, optional)
          z_U2 + z_U6 = 1                                              (competing licenses, exactly one)
          z_U4 <= z_U3                                                 (shared solvent recovery)
          sum_u z_u <= 4                                               (commissioning crew)
          x_up >= 0,  z_u in {0,1}

The hours constraint is a big-M constraint whose `M` is already tight: `H_u` is the largest value the left
side can take. The minimum throughput row is the second half of the semicontinuous pair of section 1. The
last three rows are the logical statements of section 2. The optional row is the hull-derived disaggregation
of section 4, and its effect on the bound is measured at the end.

In [ ]:
# --- Plant data --------------------------------------------------------------
UNITS = ["U1 mixer-extruder", "U2 roll press", "U3 slot-die coater",
         "U4 spray dryer", "U5 calciner", "U6 calender line"]
PRODUCTS = ["anode paste", "cathode sheet", "separator film"]

unit = pd.DataFrame({
    "fixed_kUSD":  [180.0, 240.0, 300.0, 150.0, 260.0, 210.0],
    "hours_yr":    [3500.0, 3400.0, 3600.0, 3200.0, 3300.0, 3400.0],
    "min_thru_t":  [500.0, 450.0, 600.0, 350.0, 550.0, 400.0],
}, index=UNITS)

rate = pd.DataFrame([[0.42, 0.00, 0.00], [0.30, 0.55, 0.00], [0.00, 0.62, 0.48],
                     [0.36, 0.00, 0.40], [0.00, 0.50, 0.30], [0.00, 0.44, 0.52]],
                    index=UNITS, columns=PRODUCTS)          # tonnes per hour
vcost = pd.DataFrame([[286.0, 0.0, 0.0], [318.0, 271.0, 0.0], [0.0, 289.0, 331.0],
                      [305.0, 0.0, 352.0], [0.0, 262.0, 364.0], [0.0, 296.0, 318.0]],
                     index=UNITS, columns=PRODUCTS)         # USD per tonne
demand = pd.Series({"anode paste": 1800.0, "cathode sheet": 2400.0, "separator film": 1500.0})

ARCS = [(u, p) for u in UNITS for p in PRODUCTS if rate.loc[u, p] > 0]
CAPUP = {(u, p): unit.hours_yr[u]*rate.loc[u, p] for (u, p) in ARCS}

print(unit.to_string(), "\n")
print("production rate, t/h (blank means the unit cannot make that product)")
print(rate.replace(0.0, np.nan).to_string(na_rep="-"), "\n")
print("annual capacity if a unit made only that product, t/yr")
print(pd.DataFrame([[CAPUP.get((u, p), np.nan) for p in PRODUCTS] for u in UNITS],
                   index=UNITS, columns=PRODUCTS).to_string(na_rep="-"), "\n")
print("demand, t/yr:", demand.to_dict())

In [ ]:
# --- Unit selection model ----------------------------------------------------
def build_plant(strong=False, relax=False, logic=("license", "utility", "crew"), kmax=4):
    m = pyo.ConcreteModel(name="unit_selection")
    m.U = pyo.Set(initialize=UNITS)
    m.P = pyo.Set(initialize=PRODUCTS)
    m.A = pyo.Set(initialize=ARCS, dimen=2)
    m.x = pyo.Var(m.A, domain=pyo.NonNegativeReals)
    m.z = pyo.Var(m.U, domain=pyo.UnitInterval if relax else pyo.Binary)

    m.demand = pyo.Constraint(m.P, rule=lambda m, p:
        sum(m.x[u, p] for u in UNITS if (u, p) in m.A) == demand[p])
    m.hours = pyo.Constraint(m.U, rule=lambda m, u:
        sum(m.x[u, p]/rate.loc[u, p] for p in PRODUCTS if (u, p) in m.A) <= unit.hours_yr[u]*m.z[u])
    m.min_throughput = pyo.Constraint(m.U, rule=lambda m, u:
        sum(m.x[u, p] for p in PRODUCTS if (u, p) in m.A) >= unit.min_thru_t[u]*m.z[u])
    if strong:
        m.link = pyo.Constraint(m.A, rule=lambda m, u, p:
            m.x[u, p] <= min(demand[p], CAPUP[u, p])*m.z[u])
    if "license" in logic:
        m.license = pyo.Constraint(expr=m.z[UNITS[1]] + m.z[UNITS[5]] == 1)
    if "utility" in logic:
        m.utility = pyo.Constraint(expr=m.z[UNITS[3]] <= m.z[UNITS[2]])
    if "crew" in logic:
        m.crew = pyo.Constraint(expr=sum(m.z[u] for u in m.U) <= kmax)

    m.cost = pyo.Objective(expr=sum(unit.fixed_kUSD[u]*m.z[u] for u in m.U)
                           + sum(vcost.loc[u, p]*m.x[u, p]/1000.0 for (u, p) in m.A),
                           sense=pyo.minimize)
    return m

plant = build_plant(strong=True)
r_plant = milp.solve(plant)
assert r_plant.solver.termination_condition == pyo.TerminationCondition.optimal, \
    r_plant.solver.termination_condition
PLANT_COST = pyo.value(plant.cost)

alloc = pd.DataFrame([[round(pyo.value(plant.x[u, p]), 1) if (u, p) in ARCS else np.nan
                       for p in PRODUCTS] for u in UNITS], index=UNITS, columns=PRODUCTS)
alloc["total t/yr"] = alloc.sum(axis=1, numeric_only=True).round(1)
alloc["installed"] = [int(round(pyo.value(plant.z[u]))) for u in UNITS]
alloc["min thru"] = unit.min_thru_t.values
alloc["hours used"] = [round(sum(pyo.value(plant.x[u, p])/rate.loc[u, p]
                                 for p in PRODUCTS if (u, p) in ARCS), 1) for u in UNITS]
alloc["hours avail"] = unit.hours_yr.values
print(alloc.to_string(na_rep="-"))
print(f"\ntotal annualized cost : {PLANT_COST:,.3f} kUSD/yr")
print(f"  fixed  : {sum(unit.fixed_kUSD[u]*round(pyo.value(plant.z[u])) for u in UNITS):,.3f} kUSD/yr")
print(f"  variable: {sum(vcost.loc[u, p]*pyo.value(plant.x[u, p])/1000.0 for (u, p) in ARCS):,.3f} kUSD/yr")

# every discrete rule must hold at the optimum
for u in UNITS:
    z_ = round(pyo.value(plant.z[u]))
    thru = sum(pyo.value(plant.x[u, p]) for p in PRODUCTS if (u, p) in ARCS)
    hrs = sum(pyo.value(plant.x[u, p])/rate.loc[u, p] for p in PRODUCTS if (u, p) in ARCS)
    assert (z_ == 0 and thru < 1e-6) or (z_ == 1 and thru >= unit.min_thru_t[u] - 1e-6), u
    assert hrs <= unit.hours_yr[u]*z_ + 1e-6, u
assert abs(round(pyo.value(plant.z[UNITS[1]])) + round(pyo.value(plant.z[UNITS[5]])) - 1) == 0
assert round(pyo.value(plant.z[UNITS[3]])) <= round(pyo.value(plant.z[UNITS[2]]))
assert sum(round(pyo.value(plant.z[u])) for u in UNITS) <= 4
for p in PRODUCTS:
    assert abs(sum(pyo.value(plant.x[u, p]) for u in UNITS if (u, p) in ARCS) - demand[p]) < 1e-6
print("\ndemand, hours, minimum throughput and all three logical rules verified")

In [ ]:
# --- What each logical rule costs, and what the disaggregation buys ----------
logic_rows = []
for label, logic in [("no logical rules", ()), ("license only", ("license",)),
                     ("utility only", ("utility",)), ("crew only", ("crew",)),
                     ("all three", ("license", "utility", "crew"))]:
    mm = build_plant(strong=True, logic=logic)
    rr = milp.solve(mm)
    assert rr.solver.termination_condition == pyo.TerminationCondition.optimal, label
    logic_rows.append({"restriction": label, "cost kUSD/yr": round(pyo.value(mm.cost), 3),
                       "units installed": ", ".join(u.split()[0] for u in UNITS
                                                    if pyo.value(mm.z[u]) > 0.5)})
logic_df = pd.DataFrame(logic_rows)
base = logic_df.loc[0, "cost kUSD/yr"]
logic_df.insert(2, "penalty vs no rules", (logic_df["cost kUSD/yr"] - base).round(3))
print(logic_df.to_string(index=False))

plant_rows = []
for label, st in [("big-M (hours only)", False), ("disaggregated (hull-derived)", True)]:
    mi = build_plant(strong=st)
    ri = milp.solve(mi)
    assert ri.solver.termination_condition == pyo.TerminationCondition.optimal
    ml = build_plant(strong=st, relax=True)
    rl = lp.solve(ml)
    assert rl.solver.termination_condition == pyo.TerminationCondition.optimal
    zi, zl = pyo.value(mi.cost), pyo.value(ml.cost)
    plant_rows.append({"formulation": label, "rows": mi.nconstraints(),
                       "MILP optimum": round(zi, 3), "LP bound": round(zl, 3),
                       "integrality gap percent": round(100*(zi - zl)/zi, 3)})
plant_strength = pd.DataFrame(plant_rows)
print()
print(plant_strength.to_string(index=False))
assert abs(plant_strength["MILP optimum"].iloc[0] - plant_strength["MILP optimum"].iloc[1]) < 1e-6
assert plant_strength["LP bound"].iloc[1] > plant_strength["LP bound"].iloc[0], \
    "the disaggregated formulation must give a bound at least as strong"

In [ ]:
# --- Figure 4: the selected plant --------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(10.4, 3.9), constrained_layout=True)

ax = tidy(axes[0])
ypos = np.arange(len(UNITS))[::-1]
bottom = np.zeros(len(UNITS))
for k, p in enumerate(PRODUCTS):
    vals = np.array([pyo.value(plant.x[u, p]) if (u, p) in ARCS else 0.0 for u in UNITS])
    ax.barh(ypos, vals, left=bottom, color=PALETTE[k], height=0.6, label=p)
    bottom += vals
for k, u in enumerate(UNITS):
    on = pyo.value(plant.z[u]) > 0.5
    if on:
        ax.plot([unit.min_thru_t[u]], [ypos[k]], marker="|", ms=16, mew=2.2, color=PALETTE[2],
                ls="none", zorder=5)
    ax.text(2480, ypos[k], "installed" if on else "not installed", va="center", fontsize=7,
            ha="right", color=INK if on else "#8A9298")
ax.plot([], [], marker="|", ms=12, mew=2.2, color=PALETTE[2], ls="none",
        label="minimum throughput")
ax.set_yticks(ypos); ax.set_yticklabels([u.split(" ", 1)[0] + "\n" + u.split(" ", 1)[1]
                                         for u in UNITS], fontsize=7.5)
ax.set_xlabel("annual throughput, t/yr")
ax.set_xlim(0, 2500)
ax.set_title("Allocation on the installed units", fontsize=10)
ax.legend(fontsize=7.5, loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=2)

ax = tidy(axes[1])
xp = np.arange(len(plant_strength))
ax.bar(xp - 0.19, plant_strength["LP bound"], width=0.36, color=PALETTE[1], label="LP relaxation bound")
ax.bar(xp + 0.19, plant_strength["MILP optimum"], width=0.36, color=PALETTE[0], label="MILP optimum")
for k in range(len(plant_strength)):
    ax.text(xp[k], plant_strength["MILP optimum"][k]*1.02,
            f"gap {plant_strength['integrality gap percent'][k]:.2f} %", ha="center",
            fontsize=8, color=PALETTE[2])
ax.set_xticks(xp)
ax.set_xticklabels(["big-M\n(hours only)", "disaggregated\n(hull-derived)"], fontsize=8)
ax.set_ylim(0, plant_strength["MILP optimum"].max()*1.30)
ax.set_ylabel("annualized cost, kUSD/yr")
ax.set_title("Formulation strength on the plant model", fontsize=10)
ax.legend(fontsize=8, loc="upper left")

fig.suptitle(f"Week 5, Figure 4: unit selection at {PLANT_COST:,.0f} kUSD/yr",
             color=INK, fontsize=10)
plt.show()

## 7. Interpretation

**A binary is not a switch bolted onto an LP, it is the model of a disjunction.** The pair
`u <= M z`, `u >= L z` says that stream usage lies in `{0} union [L, M]`. Figure 1 draws that set in the
`(z, u)` plane: two integer-feasible pieces, a point and a segment, and a wedge between them that only the
relaxation can see. The width of that wedge is controlled entirely by `M`.

**Each layer of operating reality costs margin, and the logic costs more than the money.** The pure blending
LP earns 413,439.75 USD. Fixed charges and minimum run rates bring it to 342,092.31 USD. Of that drop of
71,347.44 USD, 47,000.00 USD is the sum of the five fixed charges, which the solver still pays because every
stream earns its keep, and the remaining 24,347.44 USD is the price of the minimum run rates, which force
straight run up to 600 t when the LP wanted only 473.09 t. The cardinality limit alone costs a further
6,553.18 USD. The single implication `z_Butane <= z_Straight_Run` costs nothing on its own, because straight
run is already on. Imposed together the two rules cost 60,730.89 USD, of which 54,177.72 USD is what the
implication adds once the cardinality limit is already in force: the solver must then choose between butane,
the cheapest octane and vapor-pressure lever at 510 USD/t, and a fourth slot in the tank farm. One Boolean sentence, correctly translated into one inequality, moves the answer more than every
quality specification combined.

**Translations must be verified, not remembered.** The truth-table harness checks each row of the table on
every assignment of its binaries. The control case, translating `a => b` as `z_a + z_b <= 1`, fails
immediately, which is the point: this is exactly the error a hurried modeler makes, and it silently forbids
the case where both are true.

**A loose big-M is expensive twice over.** On the blending model the tight `M_c = avail_c` gives an LP bound
of 379,170.50 USD against a MILP optimum of 281,361.41 USD, a gap of 34.76 percent. With `M = 50,000` the
bound rises to 411,575.46 USD, a gap of 46.28 percent, and the relaxed binaries collapse from a sum of 3.69
to 0.18: a large `M` lets a small fraction of a binary pay for a full stream. With `M = 10^7` the bound is
essentially the pure LP value. On the larger allocation instance the same effect is visible in time as well
as in the bound. All three formulations return the same optimum, but the loose one starts from a gap of
about 75 percent and needs seconds to prove optimality, while the tight one starts from about one percent
and finishes in a fraction of a second.

**The integrality tolerance turns a large M into a hole in the model.** Fixing every binary at 10^-6, which
any solver reports as zero, and setting `M = 10^10`, the LP that remains earns essentially the full
unrestricted blending margin while paying 0.047 USD of fixed charges instead of 47,000 USD. No solver bug is
involved. `M` multiplied by the integrality tolerance is the amount of activity the model gives away for
free, and it is the modeler who chooses it.

**The convex hull is the tightest linear description of a disjunction.** Figure 3 draws four relaxations of
the same two-configuration reactor. A round `M = 500` relaxes the disjunction to almost the whole box. Using
the variable bounds gives a slightly smaller region. Computing the row-wise tightest `M` by solving one small
LP per row shrinks the region by nearly an order of magnitude, and the convex hull formulation is contained
in all of them. On this small two-term disjunction the tightest big-M and the hull give the same bound for
this particular objective, because the optimal face happens to lie in the part they share, but the hull
region is strictly smaller and would separate under a different cost vector. On the allocation instance,
where the disjunction is a fixed charge on 20 units, the disaggregated form closes the gap essentially to
zero while the tight big-M leaves about one percent.

**Formulation strength is a number, and it is the number that predicts effort.** The summary table lists the
integrality gap of every model built here. The row count is not a good predictor: the disaggregated
allocation model has an order of magnitude more rows than the big-M model and solves faster, because the
tree it has to search is smaller.

**The application ties the three devices together.** The plant model uses the semicontinuous pair (hours and
minimum throughput), three logical rules, and a hull-derived disaggregation. The logical rules cost
79.32 kUSD/yr against the unrestricted selection, and they change the equipment list, not just the number.
The disaggregation is worth a smaller gap improvement here than on the allocation instance, because the
hours constraint is already tight, which is the honest general lesson: disaggregation pays when the
aggregate big-M row is loose, and costs rows when it is not.

## 8. Exercises

**Exercise 1 (introductory).** The tank farm is re-piped and can now line up five streams instead of four.
Re-solve the blending MILP and report the new margin, the new stream selection, and the value of the fifth
slot. Then explain, without solving, why the value of a sixth slot must be zero.

**Exercise 2 (introductory).** Translate the following into linear constraints on binaries and verify each
one with the truth-table harness of section 2: (a) "if the calciner runs then either the spray dryer or the
roll press must run"; (b) "the slot-die coater and the calender line may not both be idle"; (c) "at least
two of the six units must be installed"; (d) "the spray dryer runs if and only if both the mixer-extruder and
the slot-die coater run".

**Exercise 3 (intermediate).** Compute the row-wise tightest `M` for the blending model by solving one LP per
big-M row, in the style of `tightest_big_M`, instead of asserting `M_c = avail_c`. Report the values you
obtain, confirm that they match the availabilities, and then repeat the calculation after adding a constraint
that total production may not exceed 9,000 t. Which `M` values change, and why?

**Exercise 4 (intermediate).** Build the convex hull formulation of the semicontinuous stream usage in the
blending model, that is, disaggregate `u_c` into `u_c^on` and `u_c^off` with `u_c^off = 0`. Show algebraically
that it reduces to the big-M pair already in use, and explain why the two coincide here while they do not
coincide for the two-configuration reactor of section 4.

**Exercise 5 (advanced).** Extend the plant model of section 6 to two years. Demand grows by 20 percent in
year 2, a unit installed in year 1 remains available in year 2, and installing a unit in year 2 costs 15
percent more. Introduce `z_{u,t}` with the monotonicity constraint `z_{u,t} >= z_{u,t-1}` and an installation
variable `w_{u,t} = z_{u,t} - z_{u,t-1}`. Solve, report the installation schedule, and compare the
integrality gap of the big-M and disaggregated versions of the two-year model with the one-year values in
this notebook. Comment on whether the gap grows or shrinks with the number of periods.

**Exercise 6 (introductory, cross-tool).** Build the fixed-charge blending model in
`w05-fixed-charge.xlsx` with the layout given above, including the `M`
column `I20:I24`, the two semicontinuous `VALUE` columns and the two logical cells `H26` and `H28`.
Solve it with OpenSolver using CBC. Confirm that objective cell `C45` equals the margin of the
fixed-charge MILP printed in Section 1 to the cent, and that `H20:H24` selects exactly the same set of
streams as the `on` column of the results table. Then delete the `bin` entry, add `$H$20:$H$24 <= 1`,
re-solve, and confirm that the value you obtain is the LP relaxation bound reported for the tight
formulation `M_c = avail_c` in Section 3. Count the dialog actions the second solve required, multiply
by the number of formulation pairs compared in Section 5, and state the result.


## 9. Takeaways

- A binary variable is the modeling device for a disjunction. Fixed charges, minimum run rates and semicontinuous variables `u in {0} union [L, M]` are all the same pattern: `u <= M z` together with `u >= L z`.
- Logical statements translate mechanically into linear inequalities, and the translation can be verified exhaustively on the truth table. Do that verification: `a => b` is `z_a <= z_b`, never `z_a + z_b <= 1`.
- Every valid big-M gives the same integer optimum. Only a tight one gives a usable bound. The tightest valid `M` for a row is the largest violation of that row over the rest of the feasible set, and it is worth one small LP to compute.
- A large `M` fails twice: it collapses the LP relaxation, and it multiplies the solver integrality tolerance into a quantity of free activity. `M = 10^10` with a tolerance of `10^-6` grants 10,000 units of flow through a stream the solution reports as off.
- The convex hull (disaggregated) formulation of a disjunction has the tightest possible linear relaxation. It costs one copy of the continuous variables per alternative, and that price is usually worth paying.
- Formulation strength is measurable as the integrality gap. It, not the row count, is what predicts branch-and-bound effort. Report it with every MILP result.